# InterPro — Protein Family and Domain Classification

InterPro integrates predictive models from **13 member databases** to classify proteins into families and predict domains and functional sites. Reference: Paysan-Lafosse et al. 2023, *Nucleic Acids Research* (https://doi.org/10.1093/nar/gkac993).

| Member Database | Focus |
|---|---|
| Pfam | Protein families and domains (HMM-based) |
| PANTHER | Protein family/subfamily classification + function |
| HAMAP | High-quality manual annotation of microbial proteomes |
| ProSite | Protein sites, patterns, and profiles |
| PRINTS | Fingerprints (groups of conserved motifs) |
| ProDom | Protein domain families (sequence clusters) |
| SMART | Signalling, extracellular, and chromatin-associated domains |
| TIGRFAMs | Protein families for microbial genomes (HMM-based) |
| PIRSF | Protein classification system (whole-protein families) |
| SUPERFAMILY | SCOP superfamily structural/evolutionary classification |
| Gene3D | Domain structures based on CATH structural classifications |
| CDD | Conserved Domain Database (NCBI, structure-based) |
| SFLD | Structure-Function Linkage Database (enzyme superfamilies) |

In [ ]:
import requests
import time
import re
import json
from pathlib import Path

import polars as pl
import pandas as pd

## TODO

### Ingest data
- [x] Connect to InterPro REST API and verify connectivity
- [x] Download entry metadata for all InterPro entries (type, name, member DBs, GO terms)
- [x] Paginate through full `/entry/interpro/` endpoint and save to JSON
- [x] Parse entry metadata into a flat Polars DataFrame
- [x] Fetch domain annotations for human proteome (taxon 9606)
- [x] Paginate through full `/protein/uniprot/taxonomy/9606/` endpoint and save to JSON

### Explore and clean
- [ ] Entry type distribution: family / domain / homologous superfamily / repeat / site
- [ ] Coverage statistics: fraction of human proteins with at least one InterPro annotation
- [ ] Member DB breakdown: how many entries come from each of the 13 member databases

### Domain architecture analysis
- [ ] Build domain co-occurrence matrix across human proteins
- [ ] Find common domain combinations (most frequent multi-domain architectures)

### Functional annotation
- [ ] Map GO terms to entries; summarise GO slim categories
- [ ] Pathway enrichment for domain-containing proteins

### Visualization
- [ ] Domain architecture diagrams for selected proteins
- [ ] Entry type pie chart
- [ ] Member DB bar chart (entries per source DB)

### Statistical analysis
- [ ] Domain frequency distributions across the proteome
- [ ] Power law fitting for domain usage (Zipf-like behaviour)

## 1. Ingest Data

### 1.1 Connect to InterPro API

In [ ]:
INTERPRO_BASE = "https://www.ebi.ac.uk/interpro/api"


def interpro_get(endpoint: str, params: dict = None) -> dict:
    """Perform a GET request against the InterPro REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to INTERPRO_BASE, e.g. ``"/entry/interpro/"``
        OR an absolute URL (used when following ``next`` pagination links).
    params : dict, optional
        Query parameters to append to the request URL.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.

    Notes
    -----
    A 1-second sleep is inserted after every request to stay within the
    EBI fair-use rate limit (~10 req/s recommended ceiling).
    """
    # Build the full URL: allow absolute URLs for pagination ``next`` links
    url = endpoint if endpoint.startswith("http") else f"{INTERPRO_BASE}{endpoint}"
    response = requests.get(url, params=params)
    response.raise_for_status()          # raise on HTTP errors
    time.sleep(1)                        # polite rate-limiting
    return response.json()


# --- connectivity check ---
# Fetch a single entry to confirm the API is reachable and get the total count
probe = interpro_get("/entry/interpro/", params={"page_size": 1})
total_entries = probe["count"]
print(f"InterPro API reachable. Total InterPro entries: {total_entries:,}")

### 1.2 Download InterPro Entry Metadata

In [ ]:
# Create the data directory if it does not already exist
DATA_DIR = Path("/Users/alice/github/elixir-of-life/interpro/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ENTRIES_FILE = DATA_DIR / "interpro_entries.json"

if ENTRIES_FILE.exists():
    # Skip download if the file was already saved in a previous run
    print(f"Entries file already exists: {ENTRIES_FILE}")
    with open(ENTRIES_FILE) as fh:
        all_entries = json.load(fh)
    print(f"Loaded {len(all_entries):,} entries from disk.")
else:
    all_entries = []
    # Request 200 entries per page with GO terms included as an extra field
    next_url = "/entry/interpro/"
    params = {"page_size": 200, "extra_fields": "go_terms"}
    page = 0

    while next_url:
        page += 1
        data = interpro_get(next_url, params=params)
        # After the first request we follow absolute ``next`` URLs; clear params
        params = None
        results = data.get("results", [])
        all_entries.extend(results)
        # ``next`` is None on the last page
        next_url = data.get("next")
        print(f"  Page {page:>4d} — fetched {len(results):>3d} entries "
              f"(running total: {len(all_entries):,})", end="\r")

    print()  # newline after carriage-return progress line
    # Persist to disk so we never re-download accidentally
    with open(ENTRIES_FILE, "w") as fh:
        json.dump(all_entries, fh)
    print(f"Saved {len(all_entries):,} entries → {ENTRIES_FILE}")

### 1.3 Parse Entry Metadata into DataFrame

In [ ]:
def parse_entry(record: dict) -> dict:
    """Flatten a single InterPro entry JSON record into a plain dict.

    Parameters
    ----------
    record : dict
        Raw JSON object from the ``/entry/interpro/`` API response.

    Returns
    -------
    dict
        Flat dictionary with keys: ``accession``, ``name``, ``type``,
        ``short_name``, ``member_databases``, ``go_term_count``.
    """
    metadata = record.get("metadata", {})
    # member_databases is a dict keyed by DB name; count the number of DBs
    member_dbs = metadata.get("member_databases") or {}
    # go_terms lives inside extra_fields when requested
    go_terms = (record.get("extra_fields") or {}).get("go_terms") or []
    return {
        "accession":        metadata.get("accession", ""),
        "name":             metadata.get("name", ""),
        "type":             metadata.get("type", ""),
        "short_name":       metadata.get("short_name", ""),
        "member_databases": len(member_dbs),   # integer count of contributing DBs
        "go_term_count":    len(go_terms),      # number of GO terms associated
    }


# Build the Polars DataFrame from the list of flattened records
rows = [parse_entry(r) for r in all_entries]
df_entries = pl.DataFrame(rows)

print(f"DataFrame shape: {df_entries.shape}")
df_entries.head(5)

### 1.4 Fetch Human Proteome Domain Annotations

In [ ]:
HUMAN_PROTEINS_FILE = DATA_DIR / "interpro_human_proteins.json"

if HUMAN_PROTEINS_FILE.exists():
    print(f"Human proteins file already exists: {HUMAN_PROTEINS_FILE}")
    with open(HUMAN_PROTEINS_FILE) as fh:
        all_human_proteins = json.load(fh)
    print(f"Loaded {len(all_human_proteins):,} protein records from disk.")
else:
    all_human_proteins = []
    # Taxon 9606 = Homo sapiens; entry_subset gives the InterPro entries hit by each protein
    next_url = "/protein/uniprot/taxonomy/9606/"
    params = {"page_size": 200, "extra_fields": "entry_subset"}
    page = 0

    while next_url:
        page += 1
        data = interpro_get(next_url, params=params)
        params = None                          # subsequent pages use absolute next URL
        results = data.get("results", [])
        all_human_proteins.extend(results)
        next_url = data.get("next")
        print(f"  Page {page:>4d} — fetched {len(results):>3d} proteins "
              f"(running total: {len(all_human_proteins):,})", end="\r")

    print()
    with open(HUMAN_PROTEINS_FILE, "w") as fh:
        json.dump(all_human_proteins, fh)
    print(f"Saved {len(all_human_proteins):,} human protein records → {HUMAN_PROTEINS_FILE}")

## DataFrame Column Descriptions

### `df_entries` — InterPro entry metadata

| Column | Type | Description |
|---|---|---|
| `accession` | `str` | Unique InterPro accession, e.g. `IPR000001` |
| `name` | `str` | Full descriptive name of the entry |
| `type` | `str` | Entry type: one of `family`, `domain`, `homologous_superfamily`, `repeat`, `site` |
| `short_name` | `str` | Compact identifier / slug used internally and in visualisations |
| `member_databases` | `int` | Number of contributing member databases (1–13) whose models are integrated into this entry |
| `go_term_count` | `int` | Number of Gene Ontology terms associated with this entry; 0 if unannotated |

### `all_human_proteins` — raw human proteome annotation records

Each element is a JSON object with:

| Key | Description |
|---|---|
| `metadata.accession` | UniProt accession of the protein |
| `metadata.name` | Protein name |
| `metadata.length` | Amino-acid sequence length |
| `metadata.source_organism` | Taxonomy dict (`taxId`, `fullName`) |
| `extra_fields.entry_subset` | List of InterPro entries that match this protein, each with `accession`, `entry_type`, and `entry_protein_locations` (start/end residue positions of each domain hit) |